## Goal

Understand the state that Microsoft Foundry Hosted Agents preserve at the
runtime/session layer.

We will distinguish:

- Hosted Agent session identity
- session-scoped filesystem state
- conversation history
- LangGraph checkpoint state

We will not add SQLite or Postgres persistence yet.

The core mental model is:

Foundry session
    = isolated runtime sandbox + persisted filesystem

LangGraph thread
    = durable agent execution state

Responses conversation
    = protocol-level conversation history
    

## Four kinds of state

```
1. Responses conversation
   messages / tool calls / responses

2. Hosted Agent session
   sandbox + persisted filesystem

3. LangGraph thread/checkpoint
   messages + graph state + execution progress

4. Long-term memory
   learned/retrieved knowledge across threads
```

Conversation ≠ Session ≠ Thread ≠ Long-term memory

In [1]:
import os
from pathlib import Path

print("HOME:", os.getenv("HOME"))
print("cwd:", Path.cwd())

HOME: None
cwd: c:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\notebooks


## Persistence does not mean the VM/container stays alive

Foundry may deprovision the session's compute after it becomes idle.

The persisted session filesystem survives.

When the session is resumed, Foundry restores that state onto available
compute.

```
session active

sandbox
├── /home/session/research.md
└── /home/session/data.json

       ↓ idle

compute disappears
files persisted by Foundry

       ↓ later request

new/restored sandbox
├── /home/session/research.md
└── /home/session/data.json
```

In [4]:
from pathlib import Path
import os


def get_session_home() -> Path:
    return Path(os.getenv("HOME", ".")).resolve()


session_home = get_session_home()

print(session_home)

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\notebooks


In [5]:
workspace = session_home / "deep-agent-workspace"
workspace.mkdir(parents=True, exist_ok=True)

print(workspace)

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\notebooks\deep-agent-workspace


In [6]:
note_path = workspace / "research-note.txt"

note_path.write_text(
    "Notebook 07: Hosted Agent session filesystem experiment.",
    encoding="utf-8",
)

print(note_path)
print(note_path.read_text(encoding="utf-8"))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\notebooks\deep-agent-workspace\research-note.txt
Notebook 07: Hosted Agent session filesystem experiment.


## Deep Agents already has a filesystem abstraction

Deep Agents may externalize:

- plans
- research notes
- intermediate artifacts
- large context
- subagent outputs

The Hosted Agent runtime also provides a persisted session filesystem.

Therefore an important future design question is:

> Should Deep Agents' filesystem backend use the Foundry session filesystem?

SESSION FILESYSTEM

good for:
- documents
- intermediate research
- generated artifacts
- large blobs/text
- working files


LANGGRAPH CHECKPOINTER

good for:
- messages
- graph state
- node progress
- interrupts
- resumability
- execution metadata

## Sessions are isolated

Each session has its own sandbox and persisted filesystem.

Therefore:

Session A:
/home/session/report.txt

and

Session B:
/home/session/report.txt

are logically different files in isolated sandboxes.

```
                         FOUNDry Hosted Agent

Responses request
        ↓
Session sandbox
        ├── $HOME
        │     └── Deep Agent workspace/files
        │
        └── LangGraph runtime
                 ↓
            thread_id
                 ↓
          external checkpointer
          SQLite → Postgres

## Workspace is a context-management mechanism

The model does not need every intermediate artifact in its active context.

Deep Agents can externalize large intermediate work into files.

Instead of:

```
model context
  ├── 20 pages of research
  ├── 10 tool results
  ├── raw notes
  └── final question
```

we can move intermediate material into a workspace:

```
model context
  ├── concise active state
  └── references to useful files

workspace
  ├── raw-research.md
  ├── comparison.md
  └── evidence.json
```

# Architecture decision

For `deep-agents-on-foundry`:

## Foundry Hosted Session

Use for:

- isolated execution sandbox
- session-scoped workspace
- intermediate files
- uploaded working documents

## LangGraph persistence

Use for:

- messages
- graph execution state
- checkpoints
- interrupts
- durable execution

## Long-term persistence

Use external durable stores when information must outlive the Hosted Agent
session lifecycle.

We will explore LangGraph checkpoint persistence next.